In [5]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [6]:
import numpy as np
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,recall_score,f1_score, classification_report
from src.data_load import load_data
from src.preprocess import preprocess
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [3]:
features=4
qubits=int(np.log2(features))

X_train, X_test, y_train, y_test= load_data()
X_train_scale,X_test_scale, X_train_pca, X_test_pca, pca=preprocess(X_train,X_test,features)

In [ ]:
def normalize(X):
    return X/np.linalg.norm(X,axis=1, keepdims=True)


In [5]:
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler(feature_range=(0.1, 1))
X_train_amp = normalize(mm.fit_transform(X_train_pca))
X_test_amp = normalize(np.clip(mm.transform(X_test_pca), 0.1, 1))

In [6]:
def to_states(X):
    return np.array([Statevector(row).data for row in X])

S_train=to_states(X_train_amp)
S_test=to_states(X_test_amp)

In [7]:
K_train=np.abs(S_train.conj() @ S_train.T)**2
K_test=np.abs(S_test.conj() @ S_train.T)**2
print(K_train.shape, K_test.shape)

off_diagonal= K_train[~np.eye(len(K_train),dtype=bool)]
print("kernel off diagonal mean: ",off_diagonal.mean(), "std: ",off_diagonal.std())

(455, 455) (114, 455)
kernel off diagonal mean:  0.9033733973003572 std:  0.0891679072717105


In [8]:
qsvm=SVC(kernel="precomputed",C=1.0)
qsvm.fit(K_train,y_train)
y_pred_q=qsvm.predict(K_test)

print("QSVM amplitude accuracy: ",accuracy_score(y_test,y_pred_q))
print("Malignant recall:", recall_score(y_test, y_pred_q, pos_label=0))
print("F1 (macro):", f1_score(y_test, y_pred_q, average="macro"))
print(classification_report(y_test, y_pred_q, target_names=["malignant", "benign"]))

QSVM amplitude accuracy:  0.956140350877193
Malignant recall: 0.9285714285714286
F1 (macro): 0.9526381387619444
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [9]:
svm= SVC(kernel="rbf",C=1.0, gamma="scale").fit(X_train_amp, y_train)
y_pred_classical=svm.predict(X_test_amp)
print("Classical SVM (same data) accuracy:", accuracy_score(y_test, y_pred_classical))
print("Classical malignant recall:", recall_score(y_test, y_pred_classical, pos_label=0))
print("Classical F1 (macro):", f1_score(y_test, y_pred_classical, average="macro"))

Classical SVM (same data) accuracy: 0.9473684210526315
Classical malignant recall: 0.9285714285714286
Classical F1 (macro): 0.9434523809523809


In [10]:
seeds = [0, 1, 2, 3, 4]          # use [0] first to check timing
C_grid = {"C": [0.1, 1, 10, 100, 1000]}
rbf_grid = {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.001, 0.01, 0.1, 1, 10, "scale"]}

def metrics(y_true, pred):
    return (accuracy_score(y_true, pred),
            recall_score(y_true, pred, pos_label=0),
            f1_score(y_true, pred, average="macro"))

In [12]:
rows = []
for seed in seeds:
    X_train, X_test, y_train, y_test = load_data(seed)
    _, _, Xp_tr, Xp_te, _ = preprocess(X_train, X_test, features)

    mm = MinMaxScaler(feature_range=(0.1, 1))
    Xtr = normalize(mm.fit_transform(Xp_tr))
    Xte = normalize(np.clip(mm.transform(Xp_te), 0.1, 1))

    S_tr, S_te = to_states(Xtr), to_states(Xte)
    K_tr = np.abs(S_tr.conj() @ S_tr.T) ** 2
    K_te = np.abs(S_te.conj() @ S_tr.T) ** 2

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    models = {
        "QSVM amplitude": (GridSearchCV(SVC(kernel="precomputed"), C_grid, cv=cv, scoring="f1_macro").fit(K_tr, y_train), K_te),
        "RBF tuned (same input)": (GridSearchCV(SVC(kernel="rbf"), rbf_grid, cv=cv, scoring="f1_macro").fit(Xtr, y_train), Xte),
        "Poly deg-2 control": (GridSearchCV(SVC(kernel="poly", degree=2, gamma=1, coef0=0), C_grid, cv=cv, scoring="f1_macro").fit(Xtr, y_train), Xte),
        "Linear SVM, PCA 4": (GridSearchCV(SVC(kernel="linear"), C_grid, cv=cv, scoring="f1_macro").fit(Xp_tr, y_train), Xp_te),
    }
    for name, (model, X_eval) in models.items():
        a, r_, f = metrics(y_test, model.predict(X_eval))
        rows.append({"seed": seed, "model": name, "acc": a, "recall_malignant": r_, "f1": f})

    print(f"seed {seed} done", flush=True)
    pd.DataFrame(rows).to_csv("bc_amplitude.csv", index=False)

df = pd.DataFrame(rows)
print(df.groupby("model")[["acc", "recall_malignant", "f1"]].agg(["mean", "std"]).round(3).to_string())

seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
                          acc        recall_malignant            f1       
                         mean    std             mean    std   mean    std
model                                                                     
Linear SVM, PCA 4       0.965  0.006            0.938  0.013  0.962  0.007
Poly deg-2 control      0.963  0.011            0.929  0.038  0.960  0.013
QSVM amplitude          0.963  0.011            0.929  0.038  0.960  0.013
RBF tuned (same input)  0.961  0.016            0.924  0.043  0.958  0.018
